# Task 3 and Task 6 Notebook

This notebook covers:

- Task 3: Data Quality Assessment
- Task 6: Initial Data Profiling

Datasets used:

- `orders.csv`
- `logistics.csv`
- `inventory.csv`
- `vendors.csv`
- `financials.csv`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

DATA_DIR = Path('../data')
DATASETS = {
    'orders': 'orders.csv',
    'logistics': 'logistics.csv',
    'inventory': 'inventory.csv',
    'vendors': 'vendors.csv',
    'financials': 'financials.csv',
}


## Load Datasets

In [ ]:
dfs = {name: pd.read_csv(DATA_DIR / filename) for name, filename in DATASETS.items()}
list(dfs.keys())


## Task 6: Initial Data Profiling

In [ ]:
def dataset_profile(name, df):
    return pd.Series({
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
    }, name=name)

profile_overview = pd.DataFrame([dataset_profile(name, df) for name, df in dfs.items()])
profile_overview


In [ ]:
for name, df in dfs.items():
    print(f'\n{name.upper()}')
    display(df.head())
    display(df.dtypes.rename('dtype').to_frame())


In [ ]:
missing_summary = pd.concat(
    [
        pd.DataFrame({
            'dataset': name,
            'column': df.columns,
            'missing_count': df.isna().sum().values,
            'missing_pct': (df.isna().mean() * 100).round(2).values,
        })
        for name, df in dfs.items()
    ],
    ignore_index=True,
)
missing_summary.sort_values(['dataset', 'missing_pct'], ascending=[True, False]).head(30)


In [ ]:
numeric_summary = {}
for name, df in dfs.items():
    numeric_summary[name] = df.describe(include=[np.number]).T

numeric_summary['orders'].head(10)


## Task 3: Data Quality Assessment

In [ ]:
orders = dfs['orders'].copy()
logistics = dfs['logistics'].copy()
inventory = dfs['inventory'].copy()
vendors = dfs['vendors'].copy()
financials = dfs['financials'].copy()

for df, cols in [
    (orders, ['order_date', 'promised_delivery_date', 'actual_delivery_date', 'last_modified_date']),
    (logistics, ['order_date', 'promised_delivery_date', 'actual_delivery_date', 'last_modified_date']),
    (inventory, ['snapshot_date', 'last_receipt_date', 'last_issue_date']),
    (vendors, ['snapshot_month', 'contract_start_date', 'contract_expiry_date']),
    (financials, ['record_date']),
]:
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')


In [ ]:
quality_checks = []

def add_check(dataset, issue, count, note=''):
    quality_checks.append({
        'dataset': dataset,
        'issue': issue,
        'count': int(count),
        'note': note,
    })

add_check('orders', 'negative_order_quantity', (orders['order_quantity'] < 0).sum())
add_check('orders', 'negative_unit_price', (orders['unit_price_usd'] < 0).sum())
add_check('orders', 'negative_order_value', (orders['order_value_usd'] < 0).sum())
add_check('orders', 'discount_outside_0_100', ((orders['discount_pct'] < 0) | (orders['discount_pct'] > 100)).sum())
add_check('orders', 'delivered_missing_actual_delivery', ((orders['order_status'].eq('Delivered')) & (orders['actual_delivery_date'].isna())).sum())
add_check('orders', 'actual_before_order', ((orders['actual_delivery_date'].notna()) & (orders['actual_delivery_date'] < orders['order_date'])).sum())
calc_delay = (orders['actual_delivery_date'] - orders['promised_delivery_date']).dt.days
add_check('orders', 'delay_mismatch_vs_dates', ((orders['actual_delivery_date'].notna()) & (orders['delivery_delay_days'].notna()) & (calc_delay != orders['delivery_delay_days'])).sum())
add_check('orders', 'profit_margin_missing_with_positive_order_value', ((orders['order_value_usd'] > 0) & (orders['profit_margin_pct'].isna())).sum())
add_check('orders', 'return_reason_when_not_returned_cancelled', ((orders['return_reason'].notna()) & (~orders['order_status'].isin(['Returned', 'Cancelled']))).sum())

add_check('inventory', 'available_stock_mismatch', (inventory['available_stock'] != inventory['stock_on_hand'] - inventory['units_reserved']).sum())
add_check('inventory', 'negative_stock_on_hand', (inventory['stock_on_hand'] < 0).sum())
add_check('inventory', 'negative_available_stock', (inventory['available_stock'] < 0).sum())
add_check('inventory', 'stockout_flag_mismatch', ((inventory['stockout_flag'] == 1) != (inventory['available_stock'] <= 0)).sum())
add_check('inventory', 'last_receipt_after_snapshot', (inventory['last_receipt_date'] > inventory['snapshot_date']).sum())
add_check('inventory', 'last_issue_after_snapshot', (inventory['last_issue_date'] > inventory['snapshot_date']).sum())
add_check('inventory', 'last_issue_before_last_receipt', (inventory['last_issue_date'] < inventory['last_receipt_date']).sum())
add_check('inventory', 'inventory_value_mismatch', (np.abs(inventory['total_inventory_value_usd'] - inventory['stock_on_hand'] * inventory['unit_cost_usd']) > 0.01).sum())

add_check('vendors', 'quality_acceptance_rate_missing', vendors['quality_acceptance_rate'].isna().sum())
add_check('vendors', 'financial_stability_score_missing', vendors['financial_stability_score'].isna().sum())
add_check('vendors', 'vris_score_missing', vendors['vris_score'].isna().sum())
add_check('vendors', 'rates_outside_0_100', ((vendors['on_time_delivery_rate'].lt(0)) | (vendors['on_time_delivery_rate'].gt(100)) | (vendors['quality_acceptance_rate'].dropna().lt(0)) | (vendors['quality_acceptance_rate'].dropna().gt(100))).sum())
add_check('vendors', 'contract_expiry_before_start', (vendors['contract_expiry_date'] < vendors['contract_start_date']).sum())

add_check('financials', 'duplicate_rows', financials.duplicated().sum())
add_check('financials', 'negative_revenue', (financials['revenue_usd'] < 0).sum())
add_check('financials', 'negative_procurement_cost', (financials['procurement_cost_usd'] < 0).sum())
add_check('financials', 'negative_logistics_cost', (financials['logistics_cost_usd'] < 0).sum())
add_check('financials', 'gross_profit_mismatch', (np.abs(financials['gross_profit_usd'] - (financials['revenue_usd'] - financials['procurement_cost_usd'] - financials['logistics_cost_usd'] - financials['sla_penalty_usd'])) > 0.05).sum())

quality_results = pd.DataFrame(quality_checks).sort_values(['dataset', 'count'], ascending=[True, False])
quality_results


In [ ]:
relationship_checks = pd.DataFrame([
    {
        'relationship': 'orders vs logistics exact equality',
        'result': orders.equals(logistics),
    },
    {
        'relationship': 'orders vendor ids found in vendors',
        'result': round(orders['vendor_id'].isin(vendors['vendor_id']).mean() * 100, 2),
    },
    {
        'relationship': 'inventory vendor ids found in vendors',
        'result': round(inventory['vendor_id'].isin(vendors['vendor_id']).mean() * 100, 2),
    },
    {
        'relationship': 'financial non-null order ids found in orders',
        'result': round(financials['order_id'].dropna().isin(orders['order_id']).mean() * 100, 2),
    },
    {
        'relationship': 'financial non-null shipment ids found in orders shipment ids',
        'result': round(financials['shipment_id'].dropna().isin(orders['shipment_id'].dropna()).mean() * 100, 2),
    },
])
relationship_checks


In [ ]:
def iqr_outlier_summary(df, dataset_name, columns):
    rows = []
    for column in columns:
        series = df[column].dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        outliers = ((df[column] < low) | (df[column] > high)).sum()
        rows.append({
            'dataset': dataset_name,
            'column': column,
            'outlier_count': int(outliers),
            'lower_bound': round(low, 2),
            'upper_bound': round(high, 2),
        })
    return rows

outlier_rows = []
outlier_rows += iqr_outlier_summary(orders, 'orders', ['order_value_usd', 'delivery_delay_days', 'gross_margin_usd'])
outlier_rows += iqr_outlier_summary(inventory, 'inventory', ['stock_on_hand', 'days_of_supply', 'total_inventory_value_usd'])
outlier_rows += iqr_outlier_summary(vendors, 'vendors', ['procurement_spend_usd', 'lead_time_avg_days', 'vris_score'])
outlier_rows += iqr_outlier_summary(financials, 'financials', ['revenue_usd', 'cash_flow_usd', 'financial_risk_score'])

outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary


## Final Notes

Use this notebook to support both the profiling report and the data quality assessment report. If needed, additional charts can be added later for presentation or dashboard preparation.